# Class-wise FID for matching class directories

This notebook computes one clean-fid score for each pair of directories with the same class name.

```text
real/
  class_0000/
  class_0001/
samples/
  class_0000/
  class_0001/
```

`real/class_0000` is compared only with `samples/class_0000`, and so on. Image filenames inside those directories do not need to contain class labels.


## 1. Setup

Install clean-fid in the current kernel if necessary:

```python
%pip install clean-fid
```


In [ ]:
from __future__ import annotations

import importlib.util
import json
import sys
from datetime import datetime
from pathlib import Path
from statistics import fmean

from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "compute_classwise_fid.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find compute_classwise_fid.py. Run this notebook inside the DuoDiT repository."
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from compute_classwise_fid import CleanFIDBackend, iter_image_files

print(f"Repository: {REPO_ROOT}")
print("clean-fid installed:", importlib.util.find_spec("cleanfid") is not None)


## 2. Configuration

Set the two root directories. Use `CLASSES` to evaluate only a subset, or leave it as `None` to evaluate every matching `class_*` directory.


In [ ]:
REAL_DIR = Path("/path/to/real")
SAMPLES_DIR = Path("/path/to/samples")

CLASS_PREFIX = "class_"
CLASSES = None  # Example: ["class_0000", "class_0001"]

DEVICE = "auto"  # auto, cpu, cuda, or cuda:0
BATCH_SIZE = 32
NUM_WORKERS = 4
MODE = "clean"
USE_DATAPARALLEL = True

OUTPUT_PATH = REPO_ROOT / "classwise_fid.json"
RUN_FID = False  # Set True after checking the preview.


## 3. Match corresponding directories

This cell requires the class directory names to match exactly between the real and sample roots. It also counts the images before any FID computation starts.


In [ ]:
paths_ready = REAL_DIR.is_dir() and SAMPLES_DIR.is_dir()
class_pairs = []
preview_rows = []

if not paths_ready:
    print("Set REAL_DIR and SAMPLES_DIR to existing directories, then rerun this cell.")
else:
    real_class_dirs = {
        path.name: path
        for path in REAL_DIR.iterdir()
        if path.is_dir() and path.name.startswith(CLASS_PREFIX)
    }
    sample_class_dirs = {
        path.name: path
        for path in SAMPLES_DIR.iterdir()
        if path.is_dir() and path.name.startswith(CLASS_PREFIX)
    }

    missing_samples = sorted(set(real_class_dirs) - set(sample_class_dirs))
    missing_real = sorted(set(sample_class_dirs) - set(real_class_dirs))
    if missing_samples or missing_real:
        raise ValueError(
            "Class directory mismatch. "
            f"Missing from samples: {missing_samples[:10]}; "
            f"missing from real: {missing_real[:10]}"
        )

    selected_classes = sorted(real_class_dirs) if CLASSES is None else list(dict.fromkeys(CLASSES))
    unknown_classes = [
        class_name
        for class_name in selected_classes
        if class_name not in real_class_dirs
    ]
    if unknown_classes:
        raise ValueError(f"Selected classes were not found in both roots: {unknown_classes}")

    for class_name in selected_classes:
        real_images = iter_image_files(real_class_dirs[class_name])
        sample_images = iter_image_files(sample_class_dirs[class_name])
        if len(real_images) < 2 or len(sample_images) < 2:
            raise ValueError(
                f"{class_name} needs at least 2 images per side; "
                f"found {len(real_images)} real and {len(sample_images)} samples."
            )
        class_pairs.append((class_name, real_images, sample_images))
        preview_rows.append(
            {
                "class": class_name,
                "real_images": len(real_images),
                "sample_images": len(sample_images),
            }
        )

    display(preview_rows)
    print(f"Ready to evaluate {len(class_pairs)} corresponding class pair(s).")


## 4. Compute clean-fid for each pair

After confirming the preview, set `RUN_FID = True` in the configuration cell and rerun from there. The same Inception feature extractor is reused for every class.


In [ ]:
results = []
backend = None

if not RUN_FID:
    print("FID skipped. Set RUN_FID = True after checking the directory preview.")
elif not paths_ready:
    raise ValueError("REAL_DIR and SAMPLES_DIR must point to existing directories.")
elif not class_pairs:
    raise ValueError("No class directory pairs were discovered. Run Step 3 first.")
elif importlib.util.find_spec("cleanfid") is None:
    raise ModuleNotFoundError(
        "clean-fid is not installed in this kernel. Run `%pip install clean-fid`, restart the kernel, and rerun."
    )
else:
    backend = CleanFIDBackend(
        mode=MODE,
        device=DEVICE,
        num_workers=NUM_WORKERS,
        batch_size=BATCH_SIZE,
        verbose=True,
        use_dataparallel=USE_DATAPARALLEL,
    )

    for class_name, real_images, sample_images in class_pairs:
        if min(len(real_images), len(sample_images)) < 50:
            print(f"Warning: {class_name} has a small image set; its FID may be noisy.")
        score = backend.compute(sample_images, real_images, class_name)
        results.append(
            {
                "class_name": class_name,
                "real_images": len(real_images),
                "sample_images": len(sample_images),
                "fid": score,
            }
        )
        print(f"{class_name}: FID = {score:.4f}")


## 5. Display and save results

The mean below is an unweighted average of per-class FID scores. It is not pooled FID across all images.


In [ ]:
if not results:
    print("No results to save. Run Step 4 with RUN_FID = True first.")
else:
    mean_class_fid = fmean(float(row["fid"]) for row in results)
    display(
        Markdown(
            f"### Mean FID across {len(results)} class directories: "
            f"**{mean_class_fid:.4f}**"
        )
    )

    header = "| Class | Real | Samples | FID |\n|---|---:|---:|---:|"
    rows = [
        f"| {row['class_name']} | {row['real_images']} | {row['sample_images']} | {row['fid']:.4f} |"
        for row in results
    ]
    display(Markdown("\n".join([header, *rows])))

    payload = {
        "real_dir": str(REAL_DIR.resolve()),
        "samples_dir": str(SAMPLES_DIR.resolve()),
        "mode": MODE,
        "device": str(backend.device),
        "mean_class_fid": mean_class_fid,
        "classes": results,
    }

    base_output_path = OUTPUT_PATH.expanduser().resolve()
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = base_output_path.with_name(
        f"{base_output_path.stem}_{timestamp}{base_output_path.suffix}"
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = output_path.with_suffix(output_path.suffix + ".tmp")
    temporary_path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
    temporary_path.replace(output_path)

    print(f"Saved: {output_path}")


## Notes

- Directory names must match exactly, including zero padding: `class_0001` does not match `class_1`.
- Only direct child directories beginning with `CLASS_PREFIX` are evaluated.
- Class-wise FID can be unstable for small image sets. Always report the real and sample counts with each score.
- Generated and real image counts do not need to be equal, but comparable sample counts make class-to-class interpretation easier.
